In [1]:

import pandas as pd
import numpy as np


def recode_attributes(attribute):
    if attribute == 'Non-Failing Donor':
        return 'NF'
    elif attribute == 'Peripartum cardiomyopathy (PPCM)':
        return 'PPCM'
    elif attribute == 'Dilated cardiomyopathy (DCM)':
        return 'DCM'
    elif attribute == 'Hypertrophic cardiomyopathy (HCM)':
        return 'HCM'
    else:
        return 'W'



biomart_path = '/data/bionets/og86asub/alternet-project/alternet/data/biomart.txt'
biomart = pd.read_csv(biomart_path, sep='\t')

appris_path = '/data/bionets/mi34qaba/SpliceAwareGRN/data/appris_data.appris.txt'
appris_df = pd.read_csv(appris_path, sep='\t')

# sample attributions
sample_information_path = "/data/bionets/og86asub/alternet-project/alternet/data/run_table.csv"
sample_attributes = pd.read_csv(sample_information_path)
sample_attributes = sample_attributes.loc[:, ['Run', 'etiology']]
sample_attributes['eti'] = sample_attributes['etiology'].apply(lambda x: recode_attributes(x))

# TMPs rename columns
ktmp = pd.read_csv("/data/bionets/mi34qaba/data/MAGNet/MAGNet_transcript/MAGNet_kallisto_tpms", sep = '\t')
ktmp = ktmp.rename(columns={'target_id': 'transcript_id'})
names = [x.replace('_est_tpms', '') for x in ktmp.columns]
ktmp.columns = names

# Add gene ID
ktmp = ktmp.merge(biomart.loc[: ,['Transcript stable ID', 'Gene stable ID']], left_on ='transcript_id', right_on ='Transcript stable ID')
ktmp = ktmp.drop(columns=['Transcript stable ID'])
ktmp = ktmp.rename(columns={"Gene stable ID":'gene_id'})

# subset to protein coding transcripts
ktmp = ktmp[ktmp.transcript_id.isin(appris_df[appris_df['Transcript type']=='protein_coding']['Transcript ID'])]


In [26]:

conditions = ['DCM', 'HCM', 'NF']

for condi in conditions:

    # subset data to condition
    samples = sample_attributes[sample_attributes['eti'] == condi]
    samples = samples['Run'].tolist()

    transcript_data_cp = ktmp.loc[:, ['transcript_id', 'gene_id'] + samples].copy(deep=True)

    nonzero_count = np.count_nonzero(transcript_data_cp.values, axis=1)
    frac = nonzero_count/transcript_data_cp.shape[1]
    transcript_data_cp = transcript_data_cp.iloc[np.where(frac>0.50)[0]]

    transcript_data_cp = transcript_data_cp.reset_index()
    transcript_data_cp =transcript_data_cp.drop(columns=['index'])
    print(transcript_data_cp.shape)
    print(len(transcript_data_cp.gene_id.unique()))
    print(len(transcript_data_cp.transcript_id.unique()))

    print(len(transcript_data_cp[transcript_data_cp.transcript_id.isin(tf_list['Transcript stable ID'])]['gene_id'].unique()))
    print(len(transcript_data_cp[transcript_data_cp.transcript_id.isin(tf_list['Transcript stable ID'])]['transcript_id'].unique()))

    transcript_data_cp.to_csv(f"/data/bionets/og86asub/alternet-project/alternet/data/{condi}_magnet_prefiltered_tpm.tsv", sep = '\t', index = False)


(40999, 168)
15955
40999
1539
4191
(42783, 30)
16111
42783
1549
4411
(41989, 168)
15827
41989
1530
4332
